In [ ]:
# ===== 1. Nhập nhân viên =====
Nhan_vien = []

def addEmployee(ten):
    if ten in Nhan_vien:
        print("Tên này đã có rồi:", ten)
    else:
        Nhan_vien.append(ten)

Ngay = ["Thứ 2", "Thứ 3", "Thứ 4", "Thứ 5", "Thứ 6", "Thứ 7", "Chủ nhật"]
Ca = ["Sáng", "Chiều", "Tối"]

while True:
    ten = input("Nhập tên nhân viên (bỏ trống để dừng): ")
    if ten == "":
        break
    addEmployee(ten)

print("Danh sách:", Nhan_vien)
So_nv = len(Nhan_vien)
print("Số biến dự kiến:", So_nv * len(Ngay) * len(Ca))

In [ ]:
# ===== 2. Tạo mô hình và biến =====
from ortools.sat.python import cp_model

model = cp_model.CpModel()

x = {}
for e in range(len(Nhan_vien)):
    for d in range(len(Ngay)):
        for s in range(len(Ca)):
            x[e, d, s] = model.NewBoolVar(f"x_{e}_{d}_{s}")

print("Số biến:", len(x))

In [ ]:
# ===== 3. Nhập giờ bận (chọn từ danh sách nhân viên) =====
BAN = []   # mỗi phần tử là (số thứ tự nhân viên, số thứ tự ngày, số thứ tự ca)

def chon(ten_muc, danh_sach):
    """In danh sách đánh số 1, 2, 3... rồi bắt người dùng chọn một số hợp lệ."""
    for i, ten in enumerate(danh_sach, start=1):
        print(i, "-", ten)
    while True:
        so = input(f"Chọn {ten_muc} (nhập số): ")
        if so.isdigit() and 1 <= int(so) <= len(danh_sach):
            return int(so) - 1
        print("Số không hợp lệ, nhập lại nhé.")

if len(Nhan_vien) == 0:
    print("Chưa có nhân viên nào, hãy chạy lại ô 1 trước.")
else:
    while True:
        tiep = input("Thêm giờ bận? (gõ c để thêm, Enter để dừng): ")
        if tiep == "":
            break
        e = chon("nhân viên", Nhan_vien)
        d = chon("ngày", Ngay)
        s = chon("ca", Ca)
        BAN.append((e, d, s))
        print("Đã ghi:", Nhan_vien[e], "bận", Ngay[d], "-", Ca[s])

print("Danh sách bận:", [(Nhan_vien[e], Ngay[d], Ca[s]) for e, d, s in BAN])

In [ ]:
# ===== 4. Nhập ca mong muốn (chọn từ danh sách nhân viên) =====
MONG_MUON = []   # mỗi phần tử là (số thứ tự nhân viên, số thứ tự ngày, số thứ tự ca)

if len(Nhan_vien) == 0:
    print("Chưa có nhân viên nào, hãy chạy lại ô 1 trước.")
else:
    while True:
        tiep = input("Thêm ca mong muốn? (gõ c để thêm, Enter để dừng): ")
        if tiep == "":
            break
        e = chon("nhân viên", Nhan_vien)
        d = chon("ngày", Ngay)
        s = chon("ca", Ca)
        MONG_MUON.append((e, d, s))
        print("Đã ghi:", Nhan_vien[e], "muốn làm", Ngay[d], "-", Ca[s])

print("Danh sách mong muốn:", [(Nhan_vien[e], Ngay[d], Ca[s]) for e, d, s in MONG_MUON])

In [ ]:
# ===== 5. Các luật =====
CAN_MOI_CA = 2
TOI_DA_TUAN = 10

# Luật 1: mỗi ca đủ người
for d in range(len(Ngay)):
    for s in range(len(Ca)):
        model.Add(sum(x[e, d, s] for e in range(len(Nhan_vien))) == CAN_MOI_CA)

# Luật 2: giờ bận (lấy từ ô 3)
for e, d, s in BAN:
    model.Add(x[e, d, s] == 0)

# Luật 3: mỗi người tối đa TOI_DA_TUAN ca/tuần
for e in range(len(Nhan_vien)):
    model.Add(
        sum(x[e, d, s] for d in range(len(Ngay)) for s in range(len(Ca)))
        <= TOI_DA_TUAN
    )

# Luật 4: không làm hai ca liền nhau (Sáng→Chiều→Tối→Sáng hôm sau)
so_ca_tuan = len(Ngay) * len(Ca)
for e in range(len(Nhan_vien)):
    for t in range(so_ca_tuan - 1):
        d1, s1 = divmod(t, len(Ca))
        d2, s2 = divmod(t + 1, len(Ca))
        model.Add(x[e, d1, s1] + x[e, d2, s2] <= 1)

In [ ]:
# ===== 6. Ràng buộc mềm: chia đều số ca, chia đều ca Tối, ưu tiên mong muốn =====
# tong_ca[e] = tổng số ca của nhân viên e trong tuần
tong_ca = []
for e in range(len(Nhan_vien)):
    tong = sum(x[e, d, s] for d in range(len(Ngay)) for s in range(len(Ca)))
    tong_ca.append(tong)

# Hai biến phụ: số ca của người làm NHIỀU nhất và ÍT nhất
nhieu_nhat = model.NewIntVar(0, TOI_DA_TUAN, "nhieu_nhat")
it_nhat = model.NewIntVar(0, TOI_DA_TUAN, "it_nhat")
model.AddMaxEquality(nhieu_nhat, tong_ca)
model.AddMinEquality(it_nhat, tong_ca)

# --- Chia đều ca Tối ---
S_TOI = Ca.index("Tối")
toi_ca = []
for e in range(len(Nhan_vien)):
    toi_ca.append(sum(x[e, d, S_TOI] for d in range(len(Ngay))))

toi_nhieu = model.NewIntVar(0, len(Ngay), "toi_nhieu")
toi_it = model.NewIntVar(0, len(Ngay), "toi_it")
model.AddMaxEquality(toi_nhieu, toi_ca)
model.AddMinEquality(toi_it, toi_ca)

# Trọng số: ca Tối được coi là quan trọng gấp 2 lần việc chia đều tổng số ca
TRONG_SO_TOI = 2

# --- Ưu tiên ca mong muốn ---
# Mỗi ca mong muốn được xếp trúng thì được thưởng (trừ bớt điểm phạt)
TRONG_SO_MONG = 1
duoc_mong = sum(x[e, d, s] for e, d, s in MONG_MUON)

# Mục tiêu: tổng điểm phạt càng nhỏ càng tốt
model.Minimize(
    (nhieu_nhat - it_nhat)
    + TRONG_SO_TOI * (toi_nhieu - toi_it)
    - TRONG_SO_MONG * duoc_mong
)

In [ ]:
# ===== 7. Giải và in lịch =====
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 10
trang_thai = solver.Solve(model)
print("Trạng thái:", solver.StatusName(trang_thai))

if trang_thai in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    for d in range(len(Ngay)):
        for s in range(len(Ca)):
            ten = [Nhan_vien[e] for e in range(len(Nhan_vien)) if solver.Value(x[e, d, s]) == 1]
            print(Ngay[d], "-", Ca[s], ":", ten)

    print()
    trung = sum(solver.Value(x[e, d, s]) for e, d, s in MONG_MUON)
    print("Ca mong muốn được đáp ứng:", trung, "/", len(MONG_MUON))
    print()
    print("Số ca mỗi người:")
    for e in range(len(Nhan_vien)):
        so_ca = sum(solver.Value(x[e, d, s]) for d in range(len(Ngay)) for s in range(len(Ca)))
        so_toi = sum(solver.Value(x[e, d, S_TOI]) for d in range(len(Ngay)))
        print(" ", Nhan_vien[e], ":", so_ca, "ca, trong đó", so_toi, "ca Tối")
else:
    print("Không có lịch hợp lệ. Thử giảm CAN_MOI_CA hoặc thêm nhân viên.")

In [ ]:
# ===== 8. Hàm kiểm tra lịch =====
# lich[e][d][s] = 1 nếu nhân viên e làm ngày d, ca s; ngược lại bằng 0
def kiem_tra(lich):
    """Trả về danh sách lỗi (danh sách rỗng nghĩa là lịch hợp lệ)."""
    loi = []

    # Luật 1: mỗi ca đủ người
    for d in range(len(Ngay)):
        for s in range(len(Ca)):
            so_nguoi = sum(lich[e][d][s] for e in range(len(Nhan_vien)))
            if so_nguoi != CAN_MOI_CA:
                loi.append(f"{Ngay[d]} - {Ca[s]}: có {so_nguoi} người, cần {CAN_MOI_CA}")

    # Luật 2: không xếp người vào ca bận
    for e, d, s in BAN:
        if lich[e][d][s] == 1:
            loi.append(f"{Nhan_vien[e]} bận {Ngay[d]} - {Ca[s]} nhưng vẫn bị xếp")

    # Luật 3: không quá TOI_DA_TUAN ca/tuần
    for e in range(len(Nhan_vien)):
        tong = sum(lich[e][d][s] for d in range(len(Ngay)) for s in range(len(Ca)))
        if tong > TOI_DA_TUAN:
            loi.append(f"{Nhan_vien[e]} làm {tong} ca, nhiều hơn mức tối đa {TOI_DA_TUAN}")

    # Luật 4: không làm hai ca liền nhau
    so_ca_tuan = len(Ngay) * len(Ca)
    for e in range(len(Nhan_vien)):
        for t in range(so_ca_tuan - 1):
            d1, s1 = divmod(t, len(Ca))
            d2, s2 = divmod(t + 1, len(Ca))
            if lich[e][d1][s1] == 1 and lich[e][d2][s2] == 1:
                loi.append(f"{Nhan_vien[e]} làm hai ca liền nhau: {Ngay[d1]} - {Ca[s1]} và {Ngay[d2]} - {Ca[s2]}")

    return loi


def in_ket_qua_kiem_tra(lich):
    loi = kiem_tra(lich)
    if len(loi) == 0:
        print("Lịch hợp lệ, không vi phạm luật nào.")
    else:
        print("Tìm thấy", len(loi), "lỗi:")
        for dong in loi:
            print(" -", dong)


if trang_thai in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    # Đổi kết quả của solver thành bảng 0/1
    lich = []
    for e in range(len(Nhan_vien)):
        lich_nv = []
        for d in range(len(Ngay)):
            lich_nv.append([solver.Value(x[e, d, s]) for s in range(len(Ca))])
        lich.append(lich_nv)

    print("--- Kiểm tra lịch solver vừa xếp ---")
    in_ket_qua_kiem_tra(lich)

    # Thử phá: sửa tay một ô cho sai để xem hàm có bắt được không
    import copy
    lich_hong = copy.deepcopy(lich)
    lich_hong[0][0][0] = 1 - lich_hong[0][0][0]   # đảo ô của nhân viên đầu tiên, Thứ 2 Sáng
    print()
    print("--- Kiểm tra lịch đã bị sửa sai cố ý ---")
    in_ket_qua_kiem_tra(lich_hong)